# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nancy121205/flyrank/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1: Growing pages were younger, while word count was nearly identical

The FlyRank report found that growing pages averaged 185 days old and declining pages averaged 228 days old. Average word counts were nearly the same, at 1,487 and 1,481 words, respectively.

**Methodology question:** How were growing and declining pages labelled, and were page age and trend measured over comparable observation windows? Because this is an observational comparison, the age difference does not establish that older content causes traffic decline.

### Finding 2: The relationship between freshness and growth varied by update window

The FlyRank report reported a 5.43:1 growth-to-decline ratio for pages updated 31–90 days earlier. It also reported a 37.19:1 ratio for pages in the 361+ day freshness bucket, while noting that this bucket was small and contained only 21 declining pages.

**Methodology question:** How sensitive are these ratios to the size and composition of each freshness group? Were other factors, such as content age, baseline traffic, and page selection, controlled for? The reported associations are useful for prioritization, but they do not establish that refreshing a page causes traffic to grow.


In [1]:

# Section 1: Record the two paper findings and their methodology questions.

paper_findings = [
    {
        "finding": "The FlyRank report found that growing pages averaged 185 days old and declining pages averaged 228 days old. Average word counts were nearly the same, at 1,487 and 1,481 words, respectively.",
        "question": (
            "How was the outcome defined and measured, "
            "and what observation window was used?"
        )
    },
    {
        "finding": "The FlyRank report reported a 5.43:1 growth-to-decline ratio for pages updated 31–90 days earlier. It also reported a 37.19:1 ratio for pages in the 361+ day freshness bucket, while noting that this bucket was small and contained only 21 declining pages.",
        "question": (
            "Does the validation or study design support "
            "the scope of the conclusion?"
        )
    }
]

for i, item in enumerate(paper_findings, start=1):
    print(f"Finding {i}: {item['finding']}")
    print(f"Methodology question: {item['question']}\n")

Finding 1: The FlyRank report found that growing pages averaged 185 days old and declining pages averaged 228 days old. Average word counts were nearly the same, at 1,487 and 1,481 words, respectively.
Methodology question: How was the outcome defined and measured, and what observation window was used?

Finding 2: The FlyRank report reported a 5.43:1 growth-to-decline ratio for pages updated 31–90 days earlier. It also reported a 37.19:1 ratio for pages in the 361+ day freshness bucket, while noting that this bucket was small and contained only 21 declining pages.
Methodology question: Does the validation or study design support the scope of the conclusion?



## 2. My model under an honest split (before/after)


I compare a random row split with a client-grouped split. In the random split, pages from the same client may appear in both training and testing. In the client-grouped split, each client is assigned entirely to one side.

The grouped split is my preferred validation for checking performance on unseen clients. I use the same Random Forest model, feature set, target definition, and ranking metrics for both splits.

The target is an existing decline-trend proxy based on `trend_direction == "down"`. It is not a future label. Therefore, the results measure how well the model ranks pages associated with the existing label, not how accurately it forecasts future decline.


In [2]:

# Section 2: Load data, build features, and compare validation splits.

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

RANDOM_STATE = 42

# Load the public starter CSV, preferring a local copy if available.
possible_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]

local_path = next((p for p in possible_paths if p.exists()), None)

if local_path is not None:
    df = pd.read_csv(local_path)
    print("Loaded local dataset:", local_path)
else:
    data_url = (
        "https://raw.githubusercontent.com/flyrank-bih/"
        "flyrank-ml-internship-starter/main/"
        "data/raw/content_refresh_anonymized.csv"
    )
    df = pd.read_csv(data_url)
    print("Loaded public starter dataset.")

required_columns = {
    "content_id",
    "client_id",
    "impressions_90d",
    "content_age_days",
    "trend_direction"
}

missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(
        f"Required columns missing from dataset: {sorted(missing_columns)}"
    )

# Apply the starter pipeline's eligibility filters.
df["impressions_90d"] = pd.to_numeric(
    df["impressions_90d"], errors="coerce"
)
df["content_age_days"] = pd.to_numeric(
    df["content_age_days"], errors="coerce"
)

df = df.loc[
    (df["impressions_90d"] > 0)
    & (df["content_age_days"] >= 90)
].copy()

df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)

# Existing trend label: this is NOT a future outcome.
df["target"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print(f"Eligible rows: {len(df):,}")
print(f"Unique clients: {df['client_id'].nunique():,}")
print(f"Positive-label rate: {df['target'].mean():.3f}")

if df["target"].nunique() < 2:
    raise ValueError("The target has only one class after filtering.")

# Candidate feature lists based on the starter pipeline.
numeric_candidates = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

categorical_candidates = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier"
]

numeric_features = [
    c for c in numeric_candidates if c in df.columns
]
categorical_features = [
    c for c in categorical_candidates if c in df.columns
]
features = numeric_features + categorical_features

# Explicitly forbid IDs, target fields, and existing-system scores.
forbidden_features = {
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "target",
    "is_declining_label",
    "baseline_refresh_score"
}

assert not (set(features) & forbidden_features), (
    "A forbidden field was included in the feature list."
)
assert len(features) > 0, "No candidate features were found."

X = df[features].copy()
y = df["target"].copy()
groups = df["client_id"].copy()

# Build preprocessing pipelines. Imputation and encoding are fitted
# only on each training split, avoiding preprocessing leakage.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

def make_model():
    transformers = []

    if numeric_features:
        transformers.append(
            ("numeric", numeric_pipeline, numeric_features)
        )

    if categorical_features:
        transformers.append(
            ("categorical", categorical_pipeline, categorical_features)
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            min_samples_leaf=25,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])

def precision_at_k(actual, scores, k):
    actual = np.asarray(actual)
    scores = np.asarray(scores)

    if len(actual) == 0:
        return np.nan

    k = min(k, len(actual))
    top_indices = np.argsort(scores)[::-1][:k]
    return float(actual[top_indices].mean())

def evaluate_split(name, train_indices, test_indices):
    model = make_model()

    model.fit(
        X.iloc[train_indices],
        y.iloc[train_indices]
    )

    scores = model.predict_proba(X.iloc[test_indices])[:, 1]
    actual = y.iloc[test_indices].to_numpy()

    results = {
        "split": name,
        "train_rows": len(train_indices),
        "test_rows": len(test_indices),
        "test_positive_rate": float(actual.mean()),
        "Precision@20": precision_at_k(actual, scores, 20),
        "Precision@50": precision_at_k(actual, scores, 50),
        "Average precision": (
            average_precision_score(actual, scores)
            if len(np.unique(actual)) > 1
            else np.nan
        )
    }

    return results, model, scores, actual

# BEFORE: random row split.
all_indices = np.arange(len(df))

random_train, random_test = train_test_split(
    all_indices,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

random_results, random_model, random_scores, random_actual = (
    evaluate_split(
        "Before: random row split",
        random_train,
        random_test
    )
)

# AFTER: client-grouped split.
group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

group_train, group_test = next(
    group_splitter.split(X, y, groups=groups)
)

train_clients = set(groups.iloc[group_train])
test_clients = set(groups.iloc[group_test])

assert train_clients.isdisjoint(test_clients), (
    "Client overlap detected in grouped split."
)

if (
    y.iloc[group_train].nunique() < 2
    or y.iloc[group_test].nunique() < 2
):
    raise ValueError(
        "The grouped split has only one class in training or testing. "
        "Try changing RANDOM_STATE and rerun this section."
    )

group_results, group_model, group_scores, group_actual = (
    evaluate_split(
        "After: client-grouped split",
        group_train,
        group_test
    )
)

comparison = pd.DataFrame([random_results, group_results])

print("BEFORE / AFTER VALIDATION COMPARISON")
display(comparison.round(4))

print(
    "Number of clients shared across grouped train/test:",
    len(train_clients & test_clients)
)

Loaded public starter dataset.
Eligible rows: 30,000
Unique clients: 32
Positive-label rate: 0.542
BEFORE / AFTER VALIDATION COMPARISON


,split,train_rows,test_rows,test_positive_rate,Precision@20,Precision@50,Average precision
0,Before: random row split,24000,6000,0.542,0.90,0.94,0.7671
1,After: client-grouped split,23837,6163,0.511,0.55,0.58,0.5913


Number of clients shared across grouped train/test: 0


### Interpretation

The random row split is a reference comparison. The client-grouped result is more relevant to the question of generalization to unseen clients.

I report the measured metrics and test-set positive-label rates together. A difference between the two splits may indicate that validation design matters, but it does not, by itself, prove why the scores changed. I do not claim future prediction because the target is an existing trend label.


## 3. Leakage audit

I checked the final feature list for fields that directly reveal the target, identifiers, and scores that could encode an existing decision.

The target is created from trend_direction, which is derived from trend_pct. Neither field should be included as a model feature. content_id and client_id are excluded from the features; client_id is used only for the grouped validation split.

I also considered the time-window limitation. The dataset contains trailing-90-day performance metrics, but this analysis does not prove that those features precede the existing-trend label. A future-prediction claim would require features measured before the prediction date and a separate label measured in a later, non-overlapping window.

In [3]:

# Section 3: Document the leakage audit.

leakage_audit = pd.DataFrame([
    {
        "field": "trend_direction",
        "risk": "Direct target source",
        "decision": "Excluded",
        "reason": "Used to construct the target label"
    },
    {
        "field": "trend_pct",
        "risk": "Target-derived field",
        "decision": "Excluded",
        "reason": "Used to derive trend_direction"
    },
    {
        "field": "content_id",
        "risk": "Identifier",
        "decision": "Excluded",
        "reason": "Not used as a predictive feature"
    },
    {
        "field": "client_id",
        "risk": "Identifier / grouping",
        "decision": "Excluded from features",
        "reason": "Used only to separate clients across splits"
    },
    {
        "field": "is_declining_label",
        "risk": "Target alias",
        "decision": "Excluded",
        "reason": "Represents the same label information"
    },
    {
        "field": "baseline_refresh_score",
        "risk": "Existing-system score",
        "decision": "Excluded",
        "reason": "Kept separate from model inputs for fair comparison"
    },
    {
        "field": "Trailing-90-day metrics",
        "risk": "Time-window overlap",
        "decision": "Limitation",
        "reason": (
            "The available window does not prove features precede "
            "the existing-trend label"
        )
    }
])

display(leakage_audit)

print("\nFinal model features:")
print(features)

found_forbidden = sorted(set(features) & forbidden_features)
print("\nForbidden fields in feature list:", found_forbidden)

assert not found_forbidden, (
    "Leakage audit failed: forbidden fields are in the feature list."
)

print("\nLeakage checks passed for the explicitly excluded fields.")

,field,risk,decision,reason
0,trend_direction,Direct target source,Excluded,Used to construct the target label
1,trend_pct,Target-derived field,Excluded,Used to derive trend_direction
2,content_id,Identifier,Excluded,Not used as a predictive feature
3,client_id,Identifier / grouping,Excluded from features,Used only to separate clients across splits
4,is_declining_label,Target alias,Excluded,Represents the same label information
5,baseline_refresh_score,Existing-system score,Excluded,Kept separate from model inputs for fair compa...
6,Trailing-90-day metrics,Time-window overlap,Limitation,The available window does not prove features p...



Final model features:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']

Forbidden fields in feature list: []

Leakage checks passed for the explicitly excluded fields.


## 4. Claim rewrite


**Overstated claim:** “My model predicts which pages will decline and identifies the pages that should be refreshed to increase traffic.”

**Evidence-aligned claim:** “On this dataset, my model ranked pages associated with an existing decline-trend label. I measured Precision@20 and Precision@50 on a client-grouped holdout and compared the results with a random row split. The ranked output is decision-support for human review. This analysis does not establish future decline or prove that refreshing a page will increase traffic.”

This wording describes what was measured without making a causal claim. The model's ranking can help a content team decide what to investigate first, but each recommendation still needs human review.


In [4]:

# Section 4: Summarize the measured evidence behind the rewritten claim.

claim_evidence = pd.DataFrame([
    {
        "item": "Target definition",
        "evidence": "Existing trend proxy: trend_direction == 'down'"
    },
    {
        "item": "Preferred validation",
        "evidence": "Client-grouped holdout"
    },
    {
        "item": "Grouped Precision@20",
        "evidence": round(group_results["Precision@20"], 4)
    },
    {
        "item": "Grouped Precision@50",
        "evidence": round(group_results["Precision@50"], 4)
    },
    {
        "item": "Grouped test positive-label rate",
        "evidence": round(group_results["test_positive_rate"], 4)
    },
    {
        "item": "Interpretation",
        "evidence": "Decision-support ranking for human review"
    },
    {
        "item": "Not established",
        "evidence": "Future decline or causal refresh impact"
    }
])

display(claim_evidence)

,item,evidence
0,Target definition,Existing trend proxy: trend_direction == 'down'
1,Preferred validation,Client-grouped holdout
2,Grouped Precision@20,0.55
3,Grouped Precision@50,0.58
4,Grouped test positive-label rate,0.511
5,Interpretation,Decision-support ranking for human review
6,Not established,Future decline or causal refresh impact


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.